# Advanced lab 5 — One OpenTelemetry trace in Foundry and MLflow

The correct architecture is exporter fan-out, not backend synchronization: one OpenTelemetry provider creates one trace and sends each finished span to Azure Monitor/Application Insights and an MLflow OTLP receiver. Foundry remains the operational agent view; MLflow remains the engineering, evaluation, lineage, and governance view.

Sources: [Foundry client-side tracing](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-client-side), [MLflow OTLP ingestion](https://mlflow.org/docs/latest/genai/tracing/opentelemetry/ingest/), and [Unity Catalog trace storage](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog).

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path
from urllib.parse import urlsplit
from uuid import UUID

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / 'examples' / 'foundry-curriculum')
    if (candidate / 'notebook_setup.py').is_file()
)
spec = importlib.util.spec_from_file_location(
    'foundry_curriculum_setup', curriculum_root / 'notebook_setup.py'
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

```text
Foundry agent / application
          |
   one OTel provider and trace ID
          |
     +----+----+
     |         |
Azure Monitor  OTLP/HTTP
     |         |
Foundry UI    MLflow / Databricks UC
```

Do not independently enable two instrumentation owners; that can create duplicate spans or unrelated trace IDs. Configure once in a fresh kernel, keep message-content capture off, and restart the kernel before changing tracing mode. The current preview SDK also requires `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` before instrumentation; the connected cell fails closed when that explicit feature gate is absent.

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)

foundry_probe = InMemorySpanExporter()
mlflow_probe = InMemorySpanExporter()
offline_provider = TracerProvider()
offline_provider.add_span_processor(SimpleSpanProcessor(foundry_probe))
offline_provider.add_span_processor(SimpleSpanProcessor(mlflow_probe))
offline_tracer = offline_provider.get_tracer('foundry-curriculum.dual-export')

with offline_tracer.start_as_current_span('invoke_agent') as span:
    span.set_attribute('gen_ai.operation.name', 'invoke_agent')
    span.set_attribute('gen_ai.agent.name', 'synthetic-agent')
    span.set_attribute('session.id', 'synthetic-conversation-001')
    emitted_trace_id = span.get_span_context().trace_id

offline_provider.force_flush()
foundry_trace_ids = {
    item.context.trace_id for item in foundry_probe.get_finished_spans()
}
mlflow_trace_ids = {
    item.context.trace_id for item in mlflow_probe.get_finished_spans()
}
assert foundry_trace_ids == mlflow_trace_ids == {emitted_trace_id}
{'trace_id_hex': f'{emitted_trace_id:032x}', 'exporters': 2, 'same_trace': True}

In [ ]:
backend_contracts = {
    'foundry': {
        'receiver': 'connected Application Insights via Azure Monitor exporter',
        'prerequisites': [
            'Application Insights linked to the project',
            'Log Analytics Reader for the viewer',
            'agent reference plus conversation correlation',
        ],
        'expected_delay': 'typically 2-5 minutes for client-side traces',
    },
    'oss_mlflow': {
        'transport': 'OTLP/HTTP only',
        'path': '/v1/traces',
        'routing_header': 'x-mlflow-experiment-id',
        'storage': 'SQL backend required',
    },
    'databricks_uc': {
        'transport': 'managed Databricks OTLP/HTTP endpoint',
        'routing_header': 'X-Databricks-UC-Table-Name',
        'authentication': 'platform collector/gateway with rotating keyless identity',
        'warning': (
            'never freeze a static Databricks bearer token into an agent version'
        ),
    },
}
backend_contracts

In [ ]:
RUN_DUAL_EXPORT = False


def configure_connected_dual_export():
    import httpx
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.telemetry import AIProjectInstrumentor
    from azure.identity import DefaultAzureCredential
    from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
    from opentelemetry import trace
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import (
        OTLPSpanExporter,
    )
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor

    if (
        os.environ.get('AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING', '').lower()
        != 'true'
    ):
        raise RuntimeError(
            'Set AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true before '
            'enabling the preview Foundry instrumentor.'
        )

    if not session.trace_ready:
        raise RuntimeError(
            'Configure agent name, version, and trace-correlation ID first.'
        )
    endpoint = os.environ.get('OTEL_EXPORTER_OTLP_TRACES_ENDPOINT', '').strip()
    headers_present = bool(
        os.environ.get('OTEL_EXPORTER_OTLP_TRACES_HEADERS', '').strip()
    )
    parsed = urlsplit(endpoint)
    if parsed.scheme not in {'http', 'https'} or not parsed.hostname:
        raise RuntimeError('Configure a valid OTLP/HTTP traces endpoint externally.')
    if '/api/2.0/otel/v1/traces' in parsed.path:
        raise RuntimeError(
            'Direct Databricks bearer-token export is not allowed; '
            'use the approved gateway.'
        )
    if not headers_present:
        raise RuntimeError('Configure OTLP routing/auth headers outside the notebook.')
    current = trace.get_tracer_provider()
    if type(current).__name__ != 'ProxyTracerProvider':
        raise RuntimeError('Tracing is already configured; restart the kernel first.')

    with DefaultAzureCredential() as credential, AIProjectClient(
        endpoint=session.project_endpoint, credential=credential
    ) as project:
        # This value is live telemetry configuration: never print or persist it.
        app_insights_value = (
            project.telemetry.get_application_insights_connection_string()
        )
        if '=' in app_insights_value:
            app_insights_connection = app_insights_value
        else:
            # Some linked projects return only the legacy instrumentation-key GUID.
            UUID(app_insights_value)
            resource_id = (
                session.labs.observability.application_insights_resource_id
            )
            if not session.observability_ready:
                raise RuntimeError(
                    'Configure the linked Application Insights resource ID.'
                )
            arm_token = credential.get_token(
                'https://management.azure.com/.default'
            ).token
            arm_response = httpx.get(
                f'https://management.azure.com{resource_id}',
                params={'api-version': '2020-02-02'},
                headers={'Authorization': f'Bearer {arm_token}'},
                timeout=30.0,
            )
            arm_response.raise_for_status()
            properties = arm_response.json()['properties']
            app_insights_connection = properties.get(
                'ConnectionString', properties.get('connectionString', '')
            )
            if not app_insights_connection:
                raise RuntimeError(
                    'ARM returned no Application Insights connection string.'
                )

    provider = TracerProvider(
        resource=Resource.create(
            {
                'service.name': 'foundry-curriculum-agent',
                'service.version': session.labs.agent.version,
            }
        )
    )
    provider.add_span_processor(
        BatchSpanProcessor(
            AzureMonitorTraceExporter(connection_string=app_insights_connection)
        )
    )
    # OTLPSpanExporter reads endpoint and headers from the external environment.
    provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
    trace.set_tracer_provider(provider)
    AIProjectInstrumentor().instrument(
        enable_content_recording=False,
        enable_trace_context_propagation=True,
        enable_baggage_propagation=False,
    )
    return provider


if RUN_DUAL_EXPORT:
    connected_provider = configure_connected_dual_export()
    print({'configured': True, 'content_recording': False, 'exporters': 2})
else:
    print('Connected exporters skipped; configure once in a fresh kernel.')

In [ ]:
RUN_AGENT_TRACE = False

if RUN_AGENT_TRACE:
    from opentelemetry import trace

    if not RUN_DUAL_EXPORT:
        raise RuntimeError('Configure dual export in the previous cell first.')
    tracer = trace.get_tracer('foundry-curriculum.agent')
    with tracer.start_as_current_span(
        'invoke_agent',
        attributes={
            'gen_ai.operation.name': 'invoke_agent',
            'gen_ai.agent.id': session.labs.agent.id,
            'gen_ai.agent.name': session.labs.agent.name,
        },
    ) as root_span:
        conversation, response = helpers.create_agent_response(
            session,
            'Return the word ready. This is a synthetic telemetry smoke test.',
            allow_network=True,
        )
        root_span.set_attribute('session.id', conversation.id)
        root_span.set_attribute('gen_ai.conversation.id', conversation.id)
        emitted_trace_id = root_span.get_span_context().trace_id
    connected_provider.force_flush()
    print(
        {
            'trace_id_hex': f'{emitted_trace_id:032x}',
            'conversation_id': conversation.id,
            'response_id': response.id,
            'content_recording': False,
        }
    )
else:
    print('Agent trace skipped; no telemetry or model request was sent.')

In [ ]:
agent_framework_pattern = {
    'current_api': 'agent_framework.observability.configure_otel_providers',
    'exporters': ['AzureMonitorTraceExporter', 'OTLPSpanExporter'],
    'sensitive_data': False,
    'rule': 'use this instead of creating a second global instrumentation owner',
}
agent_framework_pattern

## Verification and exit criteria

First prove that both exporters received the same trace ID locally. For a connected test, verify the Application Insights link and viewer RBAC, run one synthetic agent request, flush the provider, then wait for ingestion before searching Foundry by trace/conversation ID. Independently verify the MLflow receiver. Do not declare success because the model answered; require evidence from both destinations.

For production Databricks trace storage, select the Unity Catalog location when the experiment is created, confirm SQL warehouse and table permissions, and account for current ingestion and Private Link limitations.